<a href="https://colab.research.google.com/github/alee52/LLM_AgenticAI/blob/main/eval_fine_tuned_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q --upgrade bitsandbytes trl

!wget -q https://raw.githubusercontent.com/alee52/LLM_AgenticAI/refs/heads/main/data_prep/evaluator.py -O util.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.8 MB/s eta 0:00:00


In [2]:
import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
from peft import PeftModel

hf_token = userdata.get('HF_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    login(hf_token, add_to_git_credential=True)
    print("HuggingFace token found and set as environment variable.")
else:
    print("HF_TOKEN not found in user data. Please ensure it is set in Colab secrets.")

from util import evaluate

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HuggingFace token found and set as environment variable.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HuggingFace token found.


In [3]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "categorize_products_no_cate"
HF_USER = "leearum95" # your HF name here!

LITE_MODE = False

DATA_USER = "leearum95"
DATASET_NAME = f"{DATA_USER}/items_prompts_full_no_category"
if LITE_MODE:
  # RUN_NAME = "2026-05-16_15.51.46-lite"
  REVISION = None
else:
  # RUN_NAME = "2026-05-08_18.44.24"
  RUN_NAME = "2026-05-16_15.51.46-lite"
  REVISION = None



PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# Hyper-parameters - QLoRA

QUANT_4_BIT = True
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

In [4]:
dataset = load_dataset(DATASET_NAME)
test = dataset['test']

README.md:   0%|          | 0.00/511 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.02M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/606k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/471k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/4000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3099 [00:00<?, ? examples/s]

In [5]:
# pick the right quantization

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

In [6]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Load the fine-tuned model with PEFT
if REVISION:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
else:
  fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)


print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

Memory footprint: 2271.0 MB


In [7]:
def model_predict(item):
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs,min_new_tokens = 2, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

In [8]:
#print out max probability
def model_predict2(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = fine_tuned_model.generate(
            **inputs,
            min_new_tokens=2,
            max_new_tokens=10,
            return_dict_in_generate=True,
            output_scores=True,
        )

    output_ids = outputs.sequences
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]

    print("Generated text:")
    print(repr(tokenizer.decode(generated_ids)))

    print("\nTop 5 predicted tokens at each generated step:")

    for step, logits in enumerate(outputs.scores):
        # logits shape: [batch_size, vocab_size]
        probs = torch.softmax(logits[0], dim=-1)

        top_probs, top_token_ids = torch.topk(probs, k=5)

        chosen_token_id = generated_ids[step].item()
        chosen_text = tokenizer.decode([chosen_token_id])

        print(f"\nStep {step + 1}")
        print(f"Chosen token: {chosen_token_id} {repr(chosen_text)}")

        for prob, token_id in zip(top_probs, top_token_ids):
            token_id = token_id.item()
            token_text = tokenizer.decode([token_id])
            print(f"{token_id:>8} {repr(token_text):>15} prob={prob.item():.6f}")

    return tokenizer.decode(generated_ids)

In [9]:
size = len(test)
print(size)

3099


In [ ]:
evaluate(model_predict, test, size = len(test))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  0%|          | 0/3099 [00:00<?, ?it/s]